# 📐 Official Spider ESM + Canonical Normalization
### Evaluating the `full_system` configuration of the seeded ablation suite

This notebook computes the **Exact Set Match (ESM)** score of the fine-tuned Text-to-SQL models for **Arabic (Ar-Spider), Japanese (MultiSpider-JA) and Vietnamese (MultiSpider-VI)** using the **official `taoyds/spider` evaluator**, and measures how much a small set of **semantics-preserving SQL rewrites** (canonical normalization) changes that score.

**Why this notebook exists.** ESM compares the *structure* of the predicted and gold SQL, so two queries that return identical rows can still be scored as a mismatch when they are written differently (`IN (...)` vs an `OR` chain, `NOT IN (SELECT …)` vs `EXCEPT`, `MAX(col)` vs `ORDER BY col DESC LIMIT 1`, …). The normalization rewrites both sides into one canonical form so that ESM credits equivalent structure, and the notebook then **audits every verdict the normalization changes** against the stored execution-accuracy (EX) flag to make sure the rewrites do not over-credit wrong predictions.

**What is computed for each `(language, configuration)`:**

| # | Output | Purpose |
|---|---|---|
| 1 | Raw official ESM — overall, per difficulty (easy/medium/hard/extra), per SQL component (incl. IUEN) | Headline metric, directly comparable to published baselines |
| 2 | Canonical-normalized ESM (same rules applied to gold **and** prediction) | Structure-fair metric |
| 3 | Flipped-verdict soundness check (❌→✅ flips cross-referenced with the suite's per-sample EX flag) | Evidence that the normalization does not over-credit |
| 4 | Per-rule conversion counts and per-rule flip attribution | Which rewrite is responsible for which gain |
| 5 | Cross-check against the suite master JSON (`ESM_official`) | Confirms this environment reproduces the suite run exactly |

**Inputs.** No manual uploads are needed if the ablation suite has been run. Results are searched, in order, in:
1. the local suite folder `/content/t2s_ablation_local/<lang>/<config>.jsonl` (same session as the suite),
2. the Drive folder `<EXP_ROOT>/<lang>/<config>.jsonl`,
3. a manually uploaded `/content/<config>_<lang>.jsonl`,
4. the master `ablation_suite_<lang>.json` (Drive or `/content/`).

**Outputs.** A printed report per run, a summary table, and `esm_analysis_full_system.json` (local + copied to `EXP_ROOT` on Drive) with per-sample verdicts.

**How to run.** Execute cells 1 → 6 in order. Cell 3 lets you choose which suite configurations to evaluate (default: `full_system`; add `greedy_control` to also measure the decoder's contribution to ESM).


---
## 1️⃣ Environment setup

Installs the small set of dependencies, downloads the benchmark package, and fetches the official Spider evaluation scripts.

* **One dataset serves all three languages.** Ar-Spider, MultiSpider-JA and MultiSpider-VI all keep the *English* Spider databases, gold SQL and `tables.json`; only the natural-language questions are translated. ESM never looks at the question, so a single copy of the Spider databases is enough.
* **NLTK `punkt` is mandatory.** The official `process_sql.py` tokenizes SQL with `nltk.word_tokenize`. Without the tokenizer data every gold query silently fails to parse and ESM reads 0%, so the cell downloads `punkt`/`punkt_tab` and then runs a tokenizer smoke test that raises immediately if it is broken.
* **Official evaluator flags.** `evaluation.py` is imported in-process and the cell asserts `DISABLE_VALUE = True` and `DISABLE_DISTINCT = True`, i.e. the standard Spider ESM setting (values and `DISTINCT` are ignored) used by the published baselines.


In [ ]:
#@title 1️⃣ Setup: install dependencies, download dataset, fetch the official Spider evaluator { display-mode: "form" }
# All three benchmarks (Ar-Spider, MultiSpider-JA, MultiSpider-VI) share the
# SAME English Spider databases, gold SQL, and tables.json — so one dataset
# download serves all six evaluations.

import os, subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown", "nltk", "sqlparse"], check=True)

# NLTK tokenizer data — REQUIRED by the official process_sql.py (word_tokenize).
# If punkt is missing, every gold query fails to parse and ESM collapses to 0%.
import nltk
for pkg in ["punkt", "punkt_tab"]:
    try:
        nltk.download(pkg, quiet=True)
    except Exception as e:
        print(f"nltk download {pkg}: {e}")

# ── Ar-Spider package (contains Spider tables.json + databases) ──
if not os.path.isdir("/content/arspider"):
    print("📥 downloading dataset package…")
    subprocess.run(["gdown", "--id", "1ShZPbM2FvKQy5bh-IpLrtbFNe8LpX-ig",
                    "-O", "/content/t2s_datasets.zip"], check=True)
    subprocess.run(["unzip", "-qo", "/content/t2s_datasets.zip", "-d", "/content"], check=True)
print("dataset dir:", "OK" if os.path.isdir("/content/arspider") else "MISSING")

SPIDER_DATA_DIR = "/content/arspider"
TABLES_JSON = os.path.join(SPIDER_DATA_DIR, "tables.json")
DB_DIR = ""
for cand in ["databases", "database"]:
    p = os.path.join(SPIDER_DATA_DIR, cand)
    if os.path.isdir(p): DB_DIR = p; break
assert os.path.exists(TABLES_JSON), f"tables.json not found in {SPIDER_DATA_DIR}"
assert DB_DIR, f"database dir not found in {SPIDER_DATA_DIR}"
print("tables.json:", TABLES_JSON)
print("db dir     :", DB_DIR)

# ── Official Spider evaluation code ──
SPIDER_EVAL_DIR = "/content/spider_eval"
os.makedirs(SPIDER_EVAL_DIR, exist_ok=True)
if not os.path.exists(os.path.join(SPIDER_EVAL_DIR, "evaluation.py")):
    subprocess.run(["git","clone","--depth","1","https://github.com/taoyds/spider.git",
                    os.path.join(SPIDER_EVAL_DIR,"spider_repo")], check=True, capture_output=True)
    for fname in ["evaluation.py","process_sql.py"]:
        subprocess.run(["cp", os.path.join(SPIDER_EVAL_DIR,"spider_repo",fname),
                        os.path.join(SPIDER_EVAL_DIR,fname)], check=True)
print("evaluator  :", "OK")

# smoke-test the tokenizer BEFORE anything else — fail fast, not silently
from nltk import word_tokenize as _wt
assert _wt("SELECT count(*) FROM singer") == ["SELECT", "count", "(", "*", ")", "FROM", "singer"], \
    "NLTK tokenizer not working — rerun this cell"
print("nltk tokenizer: OK")

sys.path.insert(0, SPIDER_EVAL_DIR)
import evaluation as spider_eval
print("DISABLE_VALUE =", spider_eval.DISABLE_VALUE, "| DISABLE_DISTINCT =", spider_eval.DISABLE_DISTINCT)
assert spider_eval.DISABLE_VALUE and spider_eval.DISABLE_DISTINCT, "unexpected evaluator flags"


---
## 2️⃣ Mount Google Drive (optional)

Needed only if the suite results live on Drive (`EXP_ROOT`) or if you want the final JSON copied there. Set `MOUNT_DRIVE = False` when the results were produced in this same Colab session or were uploaded manually to `/content/`.


In [ ]:
#@title 2️⃣ (Optional) Mount Google Drive { display-mode: "form" }
MOUNT_DRIVE = True  #@param {type:"boolean"}
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')


---
## 3️⃣ Configuration and result loading

**Parameters**

| Parameter | Meaning |
|---|---|
| `EXP_ROOT` | Drive folder written by the ablation suite (`<EXP_ROOT>/<lang>/…`) |
| `CONFIGS_TO_EVAL` | Comma-separated suite configuration names. `full_system` is the headline run; adding `greedy_control` enables the greedy-vs-full comparison in cell 6 |
| `USE_INT_THRESHOLD` | Whether to include the `> n → >= n+1` rewrite. **Off by default**: under `DISABLE_VALUE` the evaluator drops literal values, so this rule can turn a genuinely wrong threshold into a match (it produced 12 EX-false flips on Japanese). Keeping it off keeps the normalization sound |

**What the loader does.** For every `(language, configuration)` pair, `load_suite_results` walks the candidate paths listed in the header, loads the per-sample records (`id`, `db_id`, `gold_sql`, `pred_sql`, `ex`), and — when a master `ablation_suite_<lang>.json` is available — also picks up the `ESM_official` value the suite stored for that configuration. That stored value is used in cell 5 as a reproducibility cross-check. Missing runs are reported and skipped rather than raising.

**Baselines.** `BASELINES` holds the best published ESM numbers for each benchmark, printed next to our scores in cell 5 for the `full_system` run.


In [ ]:
#@title 3️⃣ Config: select languages / configurations and locate the suite results { display-mode: "form" }
import os, json

EXP_ROOT = "/content/drive/MyDrive/t2s_ablation_suite"  #@param {type:"string"}
LOCAL_ROOT = "/content/t2s_ablation_local"
CONFIGS_TO_EVAL = "full_system"  #@param {type:"string"}  — comma-separated suite configuration names (e.g. "full_system,greedy_control")
LANGS = [("Arabic", "ar"), ("Japanese", "ja"), ("Vietnamese", "vi")]

# Sound normalization: int_threshold was empirically shown to over-credit
# under DISABLE_VALUE (12 EX-false flips in Japanese) and is disabled.
USE_INT_THRESHOLD = False  #@param {type:"boolean"}

def _load_jsonl_file(path):
    recs = {}
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line: continue
            try:
                r = json.loads(line); recs[r["id"]] = r
            except Exception:
                pass
    return sorted(recs.values(), key=lambda r: str(r["id"]))

def load_suite_results(lang_code, config):
    """Returns (rows, source_path, stored_esm_official) for one suite configuration.
       rows have id / db_id / gold_sql / pred_sql / ex."""
    candidates = [
        os.path.join(LOCAL_ROOT, lang_code, f"{config}.jsonl"),
        os.path.join(EXP_ROOT, lang_code, f"{config}.jsonl"),
        f"/content/{config}_{lang_code}.jsonl",
        os.path.join(EXP_ROOT, lang_code, f"ablation_suite_{lang_code}.json"),
        f"/content/ablation_suite_{lang_code}.json",
    ]
    for p in candidates:
        if not os.path.exists(p): continue
        if p.endswith(".jsonl"):
            rows = _load_jsonl_file(p)
            if rows:
                stored = None
                for mp in [os.path.join(os.path.dirname(p), f"ablation_suite_{lang_code}.json"),
                           os.path.join(EXP_ROOT, lang_code, f"ablation_suite_{lang_code}.json"),
                           f"/content/ablation_suite_{lang_code}.json"]:
                    if os.path.exists(mp):
                        try:
                            cfg = json.load(open(mp, encoding="utf-8"))["configs"].get(config)
                            if cfg: stored = cfg["metrics"].get("ESM_official")
                        except Exception:
                            pass
                        break
                return rows, p, stored
        else:
            try:
                m = json.load(open(p, encoding="utf-8"))
            except Exception:
                continue
            cfg = m.get("configs", {}).get(config)
            if cfg and cfg.get("samples"):
                return sorted(cfg["samples"], key=lambda r: str(r["id"])), p, cfg["metrics"].get("ESM_official")
    return None, None, None

RUNS = []
for lang, code in LANGS:
    for config in [c.strip() for c in CONFIGS_TO_EVAL.split(",") if c.strip()]:
        rows, src, stored = load_suite_results(code, config)
        RUNS.append((lang, code, config, rows, src, stored))
        status = f"✅ {len(rows)} samples from {src}" + (f" (suite ESM_official={stored})" if stored is not None else "") \
                 if rows else "⚠️ NOT FOUND — will skip (run/pull the ablation suite first, or upload the file)"
        print(f"{lang:11s} {config:15s}: {status}")

BASELINES = {
    "Arabic":     [("LGESQL + XLM-R + CSR (Ar-Spider best)", 66.63), ("LGESQL + XLM-R", 65.57)],
    "Japanese":   [("SAVe (MultiSpider-JA best)", 62.7), ("mRAT-SQL + GraPPa", 59.1)],
    "Vietnamese": [("SAVe (MultiSpider-VI best)", 63.1), ("mRAT-SQL + GraPPa", 60.3)],
}


---
## 4️⃣ Canonical normalizer and in-process official evaluator

This cell defines the two building blocks used by the evaluation loop.

### Normalization rules
Each rule rewrites one SQL surface pattern into a semantically equivalent canonical form. All rules are **gold-blind** (the prediction is never rewritten toward the gold — the same function is applied independently to both sides) and **conservative** (each rule bails out on nested or ambiguous cases rather than guessing).

| Rule | Rewrite | Guard |
|---|---|---|
| `in_to_or` | `col IN ('a','b')` → `col = 'a' OR col = 'b'` | Only literal lists; `IN (SELECT …)` untouched |
| `notin_to_except` | `… WHERE col NOT IN (SELECT …)` → `… EXCEPT SELECT …` | Skipped if the query already has `EXCEPT` or more than two `SELECT`s |
| `agg_to_orderby` | `SELECT MIN(col) …` → `SELECT col … ORDER BY col ASC LIMIT 1` (and `MAX` → `DESC`) | Outermost `SELECT` only; skipped if an `ORDER BY` or subquery is present |
| `between_to_and` | `col BETWEEN a AND b` → `col >= a AND col <= b` (`NOT BETWEEN` → `< a OR > b`) | — |
| `int_threshold` *(optional)* | `> n` → `>= n+1`, `< n` → `<= n-1` | Disabled by default, see cell 3 |

`canonicalize_tracked` applies the rules in order and returns both the rewritten SQL and the list of rules that fired, which is what makes the per-rule attribution in cell 5 possible.

### In-process evaluator
`esm_per_sample` reproduces the main loop of the official `evaluation.py` (`--etype match`) but returns a **per-sample** record instead of aggregate counts: the ESM verdict, the gold hardness level, parse-success flags for gold and prediction, and the per-component partial scores. Predictions that fail to parse are scored against an empty SQL tree, exactly as the official script does. Schemas and foreign-key maps are cached per database.

A final smoke test parses one simple query on a real schema and raises if the environment is not usable, so a broken setup is caught here rather than surfacing as a mysterious 0% later.


In [ ]:
#@title 4️⃣ Canonical normalizer (gold-blind rules) + in-process official ESM { display-mode: "form" }
import re, json
from collections import defaultdict

# ═══════════ CANONICAL NORMALIZATION ═══════════
# Each rule rewrites one SQL surface form into its semantically equivalent
# canonical form. Rules are applied identically to gold and prediction
# (gold-blind: the prediction is never rewritten toward the gold).

def canonical_in_to_or(sql):
    """IN (literal values) → OR chain. Does NOT touch IN (SELECT...)."""
    result = sql
    for m in re.finditer(r"(\b\w+(?:\.\w+)?)\s+IN\s*\(([^)]*)\)", sql, re.IGNORECASE):
        inner = m.group(2)
        if 'SELECT' not in inner.upper() and ('"' in inner or "'" in inner):
            col = m.group(1)
            values = re.findall(r"""['"](.*?)['"]""", inner)
            if values:
                or_parts = [f"{col} = '{v}'" for v in values]
                result = result.replace(m.group(0), " OR ".join(or_parts), 1)
    return result

def canonical_notin_to_except(sql):
    """NOT IN (SELECT...) → EXCEPT. Skips if >2 SELECTs or already has EXCEPT."""
    sql_u = sql.upper()
    if 'NOT IN' not in sql_u or 'EXCEPT' in sql_u or sql_u.count('SELECT') > 2:
        return sql
    m = re.match(
        r"(SELECT\s+.+?\s+FROM\s+.+?)\s+WHERE\s+(.*?)\s*(\w+(?:\.\w+)?)\s+NOT\s+IN\s*\(\s*(SELECT\s+.+)\)$",
        sql.strip().rstrip(';'), re.IGNORECASE | re.DOTALL
    )
    if m:
        outer = m.group(1).strip()
        other = re.sub(r'\s+(AND|OR)\s*$', '', m.group(2).strip(), flags=re.IGNORECASE).strip()
        inner = m.group(4).strip().rstrip(')')
        return f"{outer} WHERE {other} EXCEPT {inner}" if other else f"{outer} EXCEPT {inner}"
    return sql

def canonical_agg_to_orderby(sql):
    """MIN(col)/MAX(col) → ORDER BY col LIMIT 1. Only outermost SELECT, no subqueries."""
    sql_u = sql.upper()
    if ('MIN(' not in sql_u and 'MAX(' not in sql_u) or 'ORDER BY' in sql_u or sql_u.count('SELECT') > 1:
        return sql
    for agg, direction in [('MIN', 'ASC'), ('MAX', 'DESC')]:
        m = re.search(r'\b' + agg + r'\s*\(\s*(\w+(?:\.\w+)?)\s*\)', sql, re.IGNORECASE)
        if m:
            col = m.group(1)
            new_sql = sql[:m.start()] + col + sql[m.end():]
            return new_sql.rstrip(';').strip() + f" ORDER BY {col} {direction} LIMIT 1"
    return sql

def canonical_between_to_and(sql):
    result = sql
    pattern_not = r"(\b\w+(?:\.\w+)?(?:\s*\(\s*\*?\s*\))?)\s+NOT\s+BETWEEN\s+(\S+)\s+AND\s+(\S+)"
    for m in re.finditer(pattern_not, sql, re.IGNORECASE):
        col, v1, v2 = m.group(1), m.group(2), m.group(3)
        result = result.replace(m.group(0), f"({col} < {v1} OR {col} > {v2})", 1)
    pattern = r"(\b\w+(?:\.\w+)?(?:\s*\(\s*\*?\s*\))?)\s+BETWEEN\s+(\S+)\s+AND\s+(\S+)"
    for m in re.finditer(pattern, result, re.IGNORECASE):
        col, v1, v2 = m.group(1), m.group(2), m.group(3)
        result = result.replace(m.group(0), f"{col} >= {v1} AND {col} <= {v2}", 1)
    return result

def canonical_integer_threshold(sql):
    result = sql
    result = re.sub(r'(?<![=<])>\s+(\d+)\b', lambda m: f">= {int(m.group(1))+1}", result)
    result = re.sub(r'(?<![=>])<\s+(\d+)\b', lambda m: f"<= {int(m.group(1))-1}", result)
    return result

RULES = [("in_to_or", canonical_in_to_or),
         ("notin_to_except", canonical_notin_to_except),
         ("agg_to_orderby", canonical_agg_to_orderby),
         ("between_to_and", canonical_between_to_and)]
if 'USE_INT_THRESHOLD' in globals() and USE_INT_THRESHOLD:
    RULES.append(("int_threshold", canonical_integer_threshold))
print(f"normalization rules active: {[n for n,_ in RULES]}")

def canonicalize_tracked(sql):
    applied = []
    s = sql
    for name, fn in RULES:
        s1 = fn(s)
        if s1 != s:
            applied.append(name); s = s1
    return s, applied

# ═══════════ IN-PROCESS OFFICIAL EVALUATOR (mirrors evaluation.py main loop) ═══════════

EMPTY_SQL = {"except": None, "from": {"conds": [], "table_units": []}, "groupBy": [],
             "having": [], "intersect": None, "limit": None, "orderBy": [],
             "select": [False, []], "union": None, "where": []}

KMAPS = spider_eval.build_foreign_key_map_from_json(TABLES_JSON)
_schema_cache = {}
def get_schema_cached(db_id):
    if db_id not in _schema_cache:
        db_path = os.path.join(DB_DIR, db_id, db_id + ".sqlite")
        _schema_cache[db_id] = spider_eval.Schema(spider_eval.get_schema(db_path))
    return _schema_cache[db_id]

PARTIAL_TYPES = ['select','select(no AGG)','where','where(no OP)','group(no Having)',
                 'group','order','and/or','IUEN','keywords']

def esm_per_sample(pairs):
    """pairs: list of (gold_sql, pred_sql, db_id).
       Returns list of dicts: {esm, hardness, gold_parse_ok, pred_parse_ok, partial}"""
    evaluator = spider_eval.Evaluator()
    out = []
    for gold, pred, db_id in pairs:
        schema = get_schema_cached(db_id)
        kmap = KMAPS[db_id]
        rec = {"esm": 0, "hardness": None, "gold_parse_ok": True, "pred_parse_ok": True, "partial": {}}
        try:
            g_sql = spider_eval.get_sql(schema, gold)
        except Exception:
            rec["gold_parse_ok"] = False; out.append(rec); continue
        rec["hardness"] = evaluator.eval_hardness(g_sql)
        try:
            p_sql = spider_eval.get_sql(schema, pred)
        except Exception:
            rec["pred_parse_ok"] = False
            p_sql = dict(EMPTY_SQL)
        # rebuild for value/col-disabled comparison (as in evaluation.py main)
        g_valid = spider_eval.build_valid_col_units(g_sql['from']['table_units'], schema)
        g_r = spider_eval.rebuild_sql_col(g_valid, spider_eval.rebuild_sql_val(g_sql), kmap)
        p_valid = spider_eval.build_valid_col_units(p_sql['from']['table_units'], schema)
        p_r = spider_eval.rebuild_sql_col(p_valid, spider_eval.rebuild_sql_val(p_sql), kmap)
        try:
            rec["esm"] = int(evaluator.eval_exact_match(p_r, g_r))
            ps = getattr(evaluator, "partial_scores", None)
            if ps:
                rec["partial"] = {t: {"acc": ps[t]["acc"], "label_total": ps[t]["label_total"],
                                      "pred_total": ps[t]["pred_total"]} for t in PARTIAL_TYPES if t in ps}
        except Exception:
            rec["esm"] = 0
        out.append(rec)
    return out

def clean(sql):
    return " ".join(str(sql).strip().rstrip(";").split())

# ── Fail-fast smoke test: parse one simple gold on a real schema ──
_test_db = sorted(os.listdir(DB_DIR))[0]
try:
    _schema = get_schema_cached(_test_db)
    _sql = spider_eval.get_sql(_schema, "SELECT count(*) FROM " + list(_schema.schema.keys())[0])
    print(f"smoke test: parsed OK on db '{_test_db}'")
except Exception as e:
    raise RuntimeError(f"SMOKE TEST FAILED — environment broken, do not proceed: {e}")

print("✅ normalizer + in-process evaluator ready")


---
## 5️⃣ Run the evaluations

For each loaded `(language, configuration)` run the cell:

1. **Cleans** gold and predicted SQL (whitespace, trailing `;`) and builds the canonical forms, recording which rules fired on each side.
2. **Scores twice with the official evaluator** — once on the original strings (raw ESM) and once on the canonical strings (normalized ESM). A guard raises if more than 5% of *gold* queries fail to parse, which only happens when the environment is broken.
3. **Breaks raw ESM down** by gold difficulty and by SQL component (partial accuracies are averaged over samples where the component is present in gold or prediction).
4. **Audits the flipped verdicts.** Every sample that goes ❌→✅ under normalization is cross-referenced with its EX flag from the suite: a flip with `EX = True` is *sound* (the prediction really returns the gold rows); a flip with `EX = False` is a potential over-credit and is counted separately; `EX = None` means the query could not be executed. Flips are also attributed to the rule(s) that fired on that sample.
5. **Cross-checks** the raw ESM against the `ESM_official` stored in the suite master JSON. The two numbers must be identical; a mismatch means this environment differs from the one that produced the suite run.
6. For `full_system`, prints the published baselines beside our raw and normalized scores.

Everything is collected in `ALL[(language, configuration)]`, including a per-sample list, for cell 6.


In [ ]:
#@title 5️⃣ Run the evaluations { display-mode: "form" }
import json
from collections import defaultdict

ALL = {}   # (lang, mode) -> per-run record

for lang, code, mode, results, path, stored_esm in RUNS:
    if not results:
        print(f"⏭️  skip {lang} {mode}"); continue
    print("\n" + "═"*72)
    print(f"═ {lang} — {mode.upper()} ═")
    print("═"*72)
    N = len(results)
    print(f"loaded {N} samples from {path}")

    golds  = [clean(r["gold_sql"]) for r in results]
    preds  = [clean(r["pred_sql"]) for r in results]
    dbs    = [r["db_id"] for r in results]
    ex     = [r.get("ex") for r in results]
    ids    = [r.get("id") for r in results]

    # canonical forms + per-rule tracking
    g_can, g_rules, p_can, p_rules = [], [], [], []
    rule_counts = {n: {"gold":0, "pred":0} for n,_ in RULES}
    for g in golds:
        s, a = canonicalize_tracked(g); g_can.append(clean(s)); g_rules.append(a)
        for r_ in a: rule_counts[r_]["gold"] += 1
    for p in preds:
        s, a = canonicalize_tracked(p); p_can.append(clean(s)); p_rules.append(a)
        for r_ in a: rule_counts[r_]["pred"] += 1

    print("\nrunning official ESM (original)…")
    v_orig  = esm_per_sample(list(zip(golds, preds, dbs)))
    _gfail = sum(1 for v in v_orig if not v["gold_parse_ok"])
    if _gfail > 0.05 * N:
        raise RuntimeError(
            f"{_gfail}/{N} GOLD queries failed to parse — environment problem "
            f"(usually missing NLTK punkt). Re-run cells 1 and 4, then this cell.")
    print("running official ESM (canonical)…")
    v_canon = esm_per_sample(list(zip(g_can, p_can, dbs)))

    esm_o = 100*sum(v["esm"] for v in v_orig)/N
    esm_c = 100*sum(v["esm"] for v in v_canon)/N
    pred_parse_fail = sum(1 for v in v_orig if not v["pred_parse_ok"])
    gold_parse_fail = sum(1 for v in v_orig if not v["gold_parse_ok"])

    # difficulty breakdown (original)
    hard_acc = defaultdict(lambda:[0,0])
    for v in v_orig:
        if v["hardness"]:
            hard_acc[v["hardness"]][0]+=1; hard_acc[v["hardness"]][1]+=v["esm"]

    # per-component partial acc (original), incl. IUEN — applicable-sample mean
    comp = defaultdict(lambda:[0,0])
    for v in v_orig:
        for t, d in v.get("partial", {}).items():
            if d["label_total"] > 0 or d["pred_total"] > 0:
                comp[t][0]+=1; comp[t][1]+=d["acc"]

    # flipped-verdict soundness check
    flips_gain = [i for i in range(N) if v_canon[i]["esm"]==1 and v_orig[i]["esm"]==0]
    flips_loss = [i for i in range(N) if v_canon[i]["esm"]==0 and v_orig[i]["esm"]==1]
    gain_ex_true  = sum(1 for i in flips_gain if ex[i] is True)
    gain_ex_false = sum(1 for i in flips_gain if ex[i] is False)
    gain_ex_none  = sum(1 for i in flips_gain if ex[i] is None)
    rule_flip = defaultdict(lambda:[0,0])   # rule -> [flips involving it, of which EX-true]
    for i in flips_gain:
        for r_ in set(g_rules[i]) | set(p_rules[i]):
            rule_flip[r_][0]+=1; rule_flip[r_][1]+= (ex[i] is True)

    # ── report ──
    print(f"\nESM original : {esm_o:.2f}%")
    print(f"ESM canonical: {esm_c:.2f}%   Δ {esm_c-esm_o:+.2f}pp")
    print(f"pred parse failures: {pred_parse_fail} | gold parse failures: {gold_parse_fail}")
    print("\nDifficulty (original ESM):")
    for lv in ["easy","medium","hard","extra"]:
        n,c = hard_acc.get(lv,[0,0])
        if n: print(f"  {lv:7s}: {100*c/n:5.1f}%  (n={n})")
    print("\nPer-component partial acc (original, applicable samples):")
    for t in PARTIAL_TYPES:
        n,s = comp.get(t,[0,0])
        if n: print(f"  {t:18s}: {100*s/n:5.1f}%  (n={n})")
    print("\nNormalization conversions (gold / pred):")
    for n_,_ in RULES:
        print(f"  {n_:18s}: {rule_counts[n_]['gold']:4d} / {rule_counts[n_]['pred']:4d}")
    print(f"\nFlipped verdicts ❌→✅: {len(flips_gain)}   (✅→❌: {len(flips_loss)})")
    print(f"  of which EX=True : {gain_ex_true}  ({100*gain_ex_true/max(len(flips_gain),1):.1f}% sound)")
    print(f"  of which EX=False: {gain_ex_false}   ← over-credit risk cases")
    print(f"  of which EX=None : {gain_ex_none}")
    if rule_flip:
        print("  per-rule (flips / EX-true):")
        for r_,(f_,t_) in sorted(rule_flip.items(), key=lambda kv:-kv[1][0]):
            print(f"    {r_:18s}: {f_:3d} / {t_:3d}")
    if stored_esm is not None:
        agree = abs(esm_o - stored_esm) < 0.005 + 1e-9
        print(f"\nCross-check vs suite master JSON: here {esm_o:.2f} vs stored ESM_official {stored_esm:.2f} → "
              + ("✅ identical" if agree else "❌ MISMATCH — environment differs from the suite run; investigate before using these numbers"))
    if mode == "full_system":
        print("\nBaselines (ESM):")
        for name, sc in BASELINES[lang]:
            print(f"  {name:38s}: {sc:.2f}%")
        print(f"  {'Ours (original ESM)':38s}: {esm_o:.2f}%")
        print(f"  {'Ours + canonical normalization':38s}: {esm_c:.2f}%")

    ALL[(lang,mode)] = {
        "path": path, "n": N, "stored_esm_official": stored_esm,
        "esm_orig": esm_o, "esm_canon": esm_c,
        "pred_parse_fail": pred_parse_fail, "gold_parse_fail": gold_parse_fail,
        "difficulty": {k:[v[0],v[1]] for k,v in hard_acc.items()},
        "components": {k:[v[0],v[1]] for k,v in comp.items()},
        "rule_counts": rule_counts,
        "flips_gain": len(flips_gain), "flips_loss": len(flips_loss),
        "flips_gain_ex_true": gain_ex_true, "flips_gain_ex_false": gain_ex_false,
        "flips_gain_ex_none": gain_ex_none,
        "rule_flip": {k:[v[0],v[1]] for k,v in rule_flip.items()},
        "per_sample": [{"id": ids[i], "db_id": dbs[i], "ex": ex[i],
                        "esm_orig": v_orig[i]["esm"], "esm_canon": v_canon[i]["esm"],
                        "hardness": v_orig[i]["hardness"],
                        "pred_parse_ok": v_orig[i]["pred_parse_ok"],
                        "rules_pred": p_rules[i], "rules_gold": g_rules[i]} for i in range(N)],
    }

print("\n\n✅ all runs complete:", [f"{l}/{m}" for (l,m) in ALL])


---
## 6️⃣ Summary and export

Prints one compact table across all runs — raw ESM, normalized ESM, the normalization gain, number of ❌→✅ flips, the share of those flips backed by `EX = True` (**sound%**), and prediction parse failures — followed, when `greedy_control` was also evaluated, by the greedy-vs-full ESM difference (the contribution of voting + repair to the structural metric).

The complete analysis, including per-sample verdicts and rule attribution, is saved to `/content/esm_analysis_full_system.json` and copied to `EXP_ROOT` on Drive when available. This file is the single source for the ESM numbers reported in the manuscript.


In [ ]:
#@title 6️⃣ Summary table + save the analysis JSON { display-mode: "form" }
import json, shutil

print("═"*88)
print(f"{'Language':11s} {'Mode':7s} {'ESM':>7s} {'ESM+norm':>9s} {'Δnorm':>7s} {'flips':>6s} {'sound%':>7s} {'parse✗':>7s}")
print("─"*88)
for (lang,mode), r in ALL.items():
    sound = 100*r["flips_gain_ex_true"]/max(r["flips_gain"],1)
    print(f"{lang:11s} {mode:7s} {r['esm_orig']:6.2f}% {r['esm_canon']:8.2f}% {r['esm_canon']-r['esm_orig']:+6.2f} {r['flips_gain']:6d} {sound:6.1f}% {r['pred_parse_fail']:7d}")
print("═"*88)

# full-vs-greedy ESM comparison (only if greedy_control was also evaluated)
if any(m == "greedy_control" for (_, m) in ALL):
    print("\nGreedy-vs-full ESM (decoder contribution to the metric):")
    for lang in ["Arabic","Japanese","Vietnamese"]:
        v = ALL.get((lang,"full_system")); g = ALL.get((lang,"greedy_control"))
        if v and g:
            print(f"  {lang:11s}: full {v['esm_orig']:.2f}%  greedy {g['esm_orig']:.2f}%  Δ {v['esm_orig']-g['esm_orig']:+.2f}pp")

OUT = "/content/esm_analysis_full_system.json"
with open(OUT,"w",encoding="utf-8") as f:
    json.dump({f"{l}__{m}": r for (l,m),r in ALL.items()}, f, ensure_ascii=False, indent=1)
print(f"\n💾 saved → {OUT}")
try:
    shutil.copy2(OUT, os.path.join(EXP_ROOT, "esm_analysis_full_system.json"))
    print("💾 copied to Drive")
except Exception as e:
    print(f"(Drive copy skipped: {e})")
print("\n➡️  esm_analysis_full_system.json holds every number reported in the manuscript (raw ESM, normalized ESM, flips, soundness).")


---
## Reading the results

* **Raw ESM** is the number to compare with published baselines (same evaluator, same `DISABLE_VALUE`/`DISABLE_DISTINCT` setting).
* **Normalized ESM** is the structure-fair number; it should be reported together with the soundness evidence.
* **sound%** close to 100% and **EX=False flips** close to 0 indicate that the normalization only recovers structurally equivalent predictions. If `EX=False` flips appear, inspect the per-rule attribution to see which rule is responsible.
* **Cross-check ✅** confirms the suite numbers are reproduced exactly in this session; do not report numbers from a session where it shows ❌.
